# 21 — Qwen2.5-7B-Instruct few-shot (esempi recuperati dal train)

Stesso modello del notebook 07 (`qwen2.5:7b-instruct` via ollama, `max_tokens=200`,
`temperature=0.3`), ma invece della richiesta zero-shot il prompt contiene **k esempi**
(articolo → riassunto umano) presi dal **train**: per ogni cluster da riassumere si recuperano i
k cluster del train più simili (coseno fra embedding `all-mpnet-base-v2`). Slug `qwen_fewshot`.
Ambiti: `SCOPE='sample'` (smoke test) e `SCOPE='test'` (5.610 righe); `test_budgetref` non è
supportato (non ancora pilotato).

Origine: i notebook di Federica `notebooks/llm/20_preliminary_fs_test.ipynb` (esplorazione di
embedding, k e prompt) e `notebooks/llm/21_fsprompt_qwen.ipynb` (la sua corsa completa),
archiviati con i loro output. Il prompt è il suo, copiato alla lettera in `su.prompt_fewshot`;
k e variante sono fissati dal pilota del notebook 20 (su righe di **validation**, così la
scelta non guarda il test). La sua corsa non è stata importata: usava row_id locali (0..5611,
con le due righe d'intestazione Orange lette come dati) invece degli indici globali di
`complete.tab`, e non impostava il contesto di ollama.

Differenze deliberate rispetto al notebook 07, tutte dichiarate nella `config` del JSON:
- **API nativa di ollama** (`/api/chat` via `su.genera_ollama_nativo`) invece dell'endpoint
  OpenAI-compatibile: è l'unico canale che accetta `num_ctx` per richiesta. `num_ctx=32768`
  (il contesto nativo di Qwen2.5); una richiesta che satura il contesto **fallisce** invece di
  essere troncata in silenzio da ollama;
- un solo messaggio `user` (niente `system`, niente `/no_think`), come nei notebook di Federica;
- troncamenti che tengono il prompt nel contesto: articolo di ogni esempio a 1.000 parole
  (il riassunto dell'esempio resta intero), documento da riassumere a 16.000 parole
  (7 cluster del test su 5.610).

In [ ]:
# Installa le dipendenze se mancanti (per esempio su Google Colab)
try:
    import pyAutoSummarizer  # noqa: F401
except ImportError:
    %pip install pyAutoSummarizer
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    %pip install sentence-transformers

In [ ]:
# --- Configurazione ---------------------------------------------------------
import os

import summ_utils as su

METODO     = 'qwen_fewshot'
# 'sample' (default) o 'test'; scripts/run_benchmark_test.py imposta SUMM_SCOPE='test'
SCOPE      = os.environ.get('SUMM_SCOPE', 'sample')
N_SAMPLES  = 100         # deve combaciare con il campione creato dal notebook 00
SEED       = 42
# es. 3 per uno smoke test rapido; None = tutti; SUMM_LIMIT usato da run_benchmark_test.py
LIMIT      = int(os.environ['SUMM_LIMIT']) if 'SUMM_LIMIT' in os.environ else None
MODELLO    = 'qwen2.5:7b-instruct'  # tag ollama, lo stesso del notebook 07

MAX_TOKENS  = 200         # come il notebook 07
TEMPERATURE = 0.3
# Fissati dal pilota del notebook 20 (validation, 100 righe): vedi la sua conclusione
K           = 2
VARIANTE    = 'new'
NUM_CTX     = su.NUM_CTX_FEWSHOT
PAROLE_ESEMPIO   = su.PAROLE_ESEMPIO_FEWSHOT
MAX_PAROLE_DOC   = su.MAX_PAROLE_DOCUMENTO_FEWSHOT
ETICHETTA   = 'Qwen few-shot '

if su.budget_attivo(SCOPE):
    raise ValueError(f"{SCOPE!r}: l'ambito a budget non e' supportato per il few-shot")

BASE = su.trova_base_dir()
P    = su.percorsi_standard(BASE)
SAMPLE_PATH  = P['sample_dir'] / f'sample_{N_SAMPLES}_seed{SEED}.tsv'
OUT_PATH     = P['summaries_dir'] / f'{METODO}_{SCOPE}.tsv'
FEWSHOT_DIR  = BASE / 'results' / 'fewshot'
VICINI_PATH  = FEWSHOT_DIR / f'{METODO}_vicini_{SCOPE}.tsv'   # committato: il retrieval riproducibile
CACHE_EMB    = BASE / 'results' / 'embeddings_cache'          # non versionato (~140 MB)

def esempi_scope():
    """Iterabile degli esempi dell'ambito selezionato (riusato anche dalla valutazione)."""
    if SCOPE == 'sample':
        return su.carica_campione(SAMPLE_PATH)
    if SCOPE == 'test':
        return su.itera_split(P['complete_tab'], 'test')
    raise ValueError(f'SCOPE non valido: {SCOPE!r}')

print(f'Modello : {MODELLO} (API nativa ollama, num_ctx={NUM_CTX})')
print(f'Ambito  : {SCOPE} | k={K} | prompt={VARIANTE}')
print(f'Output  : {OUT_PATH}')

## Retrieval degli esempi

Gli embedding del train (44.880 cluster) si calcolano una volta e restano in cache in
`results/embeddings_cache/` (non versionata). I vicini finiscono in
`results/fewshot/qwen_fewshot_vicini_{scope}.tsv` (row_id → row_id del train, rango,
similarità), che **si committa**: a parità di file la generazione non dipende dagli embedding.
Se il TSV esiste già viene riusato; per rifare il retrieval va cancellato.

⚠️ Nell'ambito `sample` le righe del campione che vengono dal train sono **escluse**: il pool
degli esempi è l'intero train, e quelle righe ritroverebbero se stesse come esempio (è il
riassunto di riferimento dato al modello). Lo smoke test gira quindi sulle sole righe val/test
del campione.

In [ ]:
righe = [es for es in esempi_scope() if es['split'] != 'train']
print(f'{len(righe)} righe da riassumere')

if VICINI_PATH.exists():
    VICINI = su.carica_vicini(VICINI_PATH)
    print(f'Vicini dal file: {VICINI_PATH}')
else:
    VICINI = su.calcola_vicini(P['complete_tab'], righe, K, CACHE_EMB, f'{METODO}_{SCOPE}', VICINI_PATH)
assert all(len(VICINI[r['row_id']]) >= K for r in righe), 'TSV dei vicini con meno di K vicini'

ESEMPI_TRAIN = su.carica_esempi_train(P['complete_tab'],
                                      {v for r in righe for v, _ in VICINI[r['row_id']][:K]},
                                      PAROLE_ESEMPIO)
print(f'{len(ESEMPI_TRAIN)} esempi del train caricati')
sim1 = sorted(VICINI[r['row_id']][0][1] for r in righe)
print(f'Similarita\' del 1o vicino: mediana {sim1[len(sim1) // 2]:.3f}, max {sim1[-1]:.3f}')

## Generazione dei riassunti

Il modello gira **fuori dal notebook**, nel server locale di ollama (`ollama pull
qwen2.5:7b-instruct`). Il prompt di ogni riga si costruisce qui, per `row_id`, e il ciclo
condiviso riceve direttamente il prompt (`prepara` è l'identità): la mappa riga → esempi non
dipende dal testo del documento. Una risposta vuota o un contesto saturo sollevano
un'eccezione: la riga è registrata come errore e **non** scritta nel TSV (ritentabile).

⚠️ Ripresa: come negli altri notebook, rieseguire su un TSV esistente aggiunge solo le righe
mancanti. Se cambiano `K`, `VARIANTE` o i troncamenti, **cancellare prima** il TSV.

In [ ]:
import statistics

def prompt_riga(es):
    esempi = [ESEMPI_TRAIN[v] for v, _ in VICINI[es['row_id']][:K]]
    documento = su.tronca_parole(su.prepara_documento(es['document']), MAX_PAROLE_DOC)
    return su.prompt_fewshot(esempi, documento, VARIANTE)

def righe_con_prompt():
    for es in righe:
        yield {**es, 'document': prompt_riga(es)}

TOKEN_PROMPT = []   # diagnostica di questa esecuzione (non sopravvive a una ripresa)

def genera(prompt):
    testo, n_prompt = su.genera_ollama_nativo([{'role': 'user', 'content': prompt}], MODELLO,
                                              MAX_TOKENS, TEMPERATURE, num_ctx=NUM_CTX)
    TOKEN_PROMPT.append(n_prompt)
    return testo

scrittore = su.ScrittoreRiassunti(OUT_PATH)
errori = su.ciclo_summarization(righe_con_prompt(), scrittore, genera, limit=LIMIT,
                                etichetta=ETICHETTA, prepara=lambda p: p)
scrittore.chiudi()
if TOKEN_PROMPT:
    print(f'Token del prompt: mediana {statistics.median(TOKEN_PROMPT):.0f}, '
          f'max {max(TOKEN_PROMPT)} (contesto {NUM_CTX})')

## Valutazione (indipendente dalla generazione)

Legge **solo** i file salvati; rieseguibile senza rigenerare. Stesse metriche e stessa
normalizzazione di tutti i metodi del benchmark. Il BERTScore **non** si calcola qui: lo
aggiunge il notebook 13 (incrementale), il G-Eval il notebook 14 / `scripts/run_geval.py`.

In [ ]:
import json

riassunti   = su.carica_riassunti(OUT_PATH)
riferimenti = esempi_scope()
config = {'modello': MODELLO,
          'backend': 'ollama (API nativa /api/chat)',
          'max_tokens': MAX_TOKENS, 'temperature': TEMPERATURE, 'num_ctx': NUM_CTX,
          'k': K, 'variante_prompt': VARIANTE,
          'embedding': su.MODELLO_EMBEDDING_FEWSHOT, 'pool_esempi': 'split train di complete.tab',
          'parole_esempio': PAROLE_ESEMPIO, 'max_parole_documento': MAX_PAROLE_DOC,
          'vicini': VICINI_PATH.name,
          'note': 'few-shot con esempi recuperati (nearest neighbour coseno), prompt di '
                  'Federica (notebooks/llm/20); k e variante dal pilota del notebook 20'}

righe_metriche, aggregato = su.valuta_e_salva(riferimenti, riassunti, METODO, SCOPE,
                                              P['metrics_dir'], config)
print(json.dumps(aggregato['overall'], indent=2))
print('\nMedie per split:')
for split, valori in aggregato['per_split'].items():
    print(f"  {split:5s} (n={valori['n_esempi']}): ROUGE-1 F1 = {valori['rouge1_f1']:.3f}")

## Ispezione qualitativa

In [ ]:
su.mostra_esempi(esempi_scope(), riassunti, quanti=2)